In [12]:
import pandas as pd
# Read Data
data_path ="data/mmm_multidimensional_example.csv"

data_df = pd.read_csv(data_path, parse_dates=["date"]) 
print(data_df.info())
data_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     318 non-null    datetime64[ns]
 1   geo      318 non-null    object        
 2   x1       318 non-null    float64       
 3   x2       318 non-null    float64       
 4   event_1  318 non-null    int64         
 5   event_2  318 non-null    int64         
 6   y        318 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(1)
memory usage: 17.5+ KB
None


,date,geo,x1,x2,event_1,event_2,y
0,2022-06-06,geo_a,5527.640078,0.000000,0,0,2647.596355
1,2022-06-06,geo_b,8849.257500,8063.918386,0,0,682.406280
2,2022-06-13,geo_a,6692.655692,0.000000,0,0,5020.823907
3,2022-06-13,geo_b,9073.817994,9354.014585,0,0,3753.104897
4,2022-06-20,geo_a,7124.016733,0.000000,0,0,6184.322132


# Data Transformation

Rename columns and categories to match business terminology

In [13]:
import numpy as np
from scipy import stats

# Make a copy to work with
df_enhanced = data_df.copy()

# 1. Rename columns
column_mapping = {
    'y': 'NetGMV',
    'x1': 'Spend_channel_1',
    'x2': 'Spend_channel_2',
    'geo': 'Groups'
}
df_enhanced.rename(columns=column_mapping, inplace=True)

# 2. Rename geo categories
geo_mapping = {
    'geo_a': 'Brand_A',
    'geo_b': 'Brand_B'
}
df_enhanced['Groups'] = df_enhanced['Groups'].replace(geo_mapping)

print("After renaming:")
print(df_enhanced.head())
print(f"\nGroups categories: {df_enhanced['Groups'].unique()}")

After renaming:
        date   Groups  Spend_channel_1  Spend_channel_2  event_1  event_2  \
0 2022-06-06  Brand_A      5527.640078         0.000000        0        0   
1 2022-06-06  Brand_B      8849.257500      8063.918386        0        0   
2 2022-06-13  Brand_A      6692.655692         0.000000        0        0   
3 2022-06-13  Brand_B      9073.817994      9354.014585        0        0   
4 2022-06-20  Brand_A      7124.016733         0.000000        0        0   

        NetGMV  
0  2647.596355  
1   682.406280  
2  5020.823907  
3  3753.104897  
4  6184.322132  

Groups categories: ['Brand_A' 'Brand_B']


# Add Correlated Variables

Add Unpaid_visits and Paid_visits with Spearman correlation 0.4-0.6 to NetGMV

In [14]:
def create_correlated_variable(y_values, target_corr=0.5, seed=None):
    """
    Create a variable with specified Spearman correlation to y_values.
    Uses Cholesky decomposition: x = ρ·y + √(1-ρ²)·z
    
    Parameters:
    - y_values: target variable to correlate with
    - target_corr: desired Spearman correlation (0.4-0.6)
    - seed: random seed for reproducibility
    
    Returns:
    - Array with target Spearman correlation to y_values
    """
    if seed is not None:
        np.random.seed(seed)
    
    n = len(y_values)
    
    # Step 1: Convert y_values to ranks and then to standard normal
    y_ranks = stats.rankdata(y_values)
    y_uniform = y_ranks / (n + 1)  # Convert to uniform [0,1]
    y_normal = stats.norm.ppf(y_uniform)  # Convert to standard normal
    
    # Step 2: Generate independent standard normal variable
    z_normal = np.random.randn(n)
    
    # Step 3: Apply Cholesky decomposition formula
    # x = ρ·y + √(1-ρ²)·z
    rho = target_corr
    x_normal = rho * y_normal + np.sqrt(1 - rho**2) * z_normal
    
    # Step 4: Convert back through uniform to get correlated values
    x_uniform = stats.norm.cdf(x_normal)
    
    # Step 5: Map to realistic visit values (scale based on y_values range)
    y_mean = np.mean(y_values)
    y_std = np.std(y_values)
    
    # Use quantile mapping to preserve distribution shape
    sorted_y = np.sort(y_values)
    indices = (x_uniform * (n - 1)).astype(int)
    x_values = sorted_y[indices]
    
    # Scale to have similar magnitude (with some variation)
    scale = np.random.uniform(0.9, 1.1)
    x_values = x_values * scale
    
    # Ensure positive values for visit counts
    x_values = np.abs(x_values)
    
    return x_values


# Set random seed for reproducibility
np.random.seed(42)

# FIX: Pre-initialize arrays with correct size to maintain row order
n_rows = len(df_enhanced)
unpaid_visits_array = np.zeros(n_rows)
paid_visits_array = np.zeros(n_rows)

# Process each group and assign to correct indices
for group in df_enhanced['Groups'].unique():
    # Get indices for this group (maintains original row order)
    group_indices = df_enhanced[df_enhanced['Groups'] == group].index
    group_y = df_enhanced.loc[group_indices, 'NetGMV'].values
    
    # Create correlated variables
    unpaid_visits_group = create_correlated_variable(
        group_y, 
        target_corr=0.55,
        seed=42 if group == 'Brand_A' else 43
    )
    
    paid_visits_group = create_correlated_variable(
        group_y, 
        target_corr=0.50,
        seed=44 if group == 'Brand_A' else 45
    )
    
    # FIX: Use index-based assignment to maintain alignment
    unpaid_visits_array[group_indices] = unpaid_visits_group
    paid_visits_array[group_indices] = paid_visits_group

# Add to dataframe
df_enhanced['Unpaid_visits'] = unpaid_visits_array
df_enhanced['Paid_visits'] = paid_visits_array

print("New variables added with proper alignment!")
print(f"\nDataFrame shape: {df_enhanced.shape}")
print(f"\nColumns: {df_enhanced.columns.tolist()}")

New variables added with proper alignment!

DataFrame shape: (318, 9)

Columns: ['date', 'Groups', 'Spend_channel_1', 'Spend_channel_2', 'event_1', 'event_2', 'NetGMV', 'Unpaid_visits', 'Paid_visits']


# Verify Correlations

Check the Spearman correlation between new variables and NetGMV

In [15]:
# Calculate Spearman correlations
print("=" * 60)
print("OVERALL CORRELATIONS (All Groups)")
print("=" * 60)

overall_corr_unpaid = stats.spearmanr(df_enhanced['NetGMV'], df_enhanced['Unpaid_visits'])
overall_corr_paid = stats.spearmanr(df_enhanced['NetGMV'], df_enhanced['Paid_visits'])

print(f"Unpaid_visits vs NetGMV: {overall_corr_unpaid.correlation:.3f} (p-value: {overall_corr_unpaid.pvalue:.4f})")
print(f"Paid_visits vs NetGMV: {overall_corr_paid.correlation:.3f} (p-value: {overall_corr_paid.pvalue:.4f})")

print("\n" + "=" * 60)
print("GROUP-SPECIFIC CORRELATIONS")
print("=" * 60)

for group in df_enhanced['Groups'].unique():
    group_data = df_enhanced[df_enhanced['Groups'] == group]
    
    corr_unpaid = stats.spearmanr(group_data['NetGMV'], group_data['Unpaid_visits'])
    corr_paid = stats.spearmanr(group_data['NetGMV'], group_data['Paid_visits'])
    
    print(f"\n{group}:")
    print(f"  Unpaid_visits vs NetGMV: {corr_unpaid.correlation:.3f}")
    print(f"  Paid_visits vs NetGMV: {corr_paid.correlation:.3f}")
    print(f"  Number of observations: {len(group_data)}")

OVERALL CORRELATIONS (All Groups)
Unpaid_visits vs NetGMV: 0.611 (p-value: 0.0000)
Paid_visits vs NetGMV: 0.459 (p-value: 0.0000)

GROUP-SPECIFIC CORRELATIONS

Brand_A:
  Unpaid_visits vs NetGMV: 0.657
  Paid_visits vs NetGMV: 0.448
  Number of observations: 159

Brand_B:
  Unpaid_visits vs NetGMV: 0.527
  Paid_visits vs NetGMV: 0.472
  Number of observations: 159


# Final Dataset Preview

In [6]:
print("Enhanced Dataset Summary:")
print("=" * 60)
print(f"Total rows: {len(df_enhanced)}")
print(f"Date range: {df_enhanced['date'].min()} to {df_enhanced['date'].max()}")
print(f"\nColumns: {df_enhanced.columns.tolist()}")
print(f"\nGroups: {df_enhanced['Groups'].unique()}")
print(f"\nRows per group:")
print(df_enhanced.groupby('Groups').size())

print("\n" + "=" * 60)
print("Sample of enhanced dataset:")
print("=" * 60)
df_enhanced.head(10)

Enhanced Dataset Summary:
Total rows: 318
Date range: 2022-06-06 00:00:00 to 2025-06-16 00:00:00

Columns: ['date', 'Groups', 'Spend_channel_1', 'Spend_channel_2', 'event_1', 'event_2', 'NetGMV', 'Unpaid_visits', 'Paid_visits']

Groups: ['Brand_A' 'Brand_B']

Rows per group:
Groups
Brand_A    159
Brand_B    159
dtype: int64

Sample of enhanced dataset:


,date,Groups,Spend_channel_1,Spend_channel_2,event_1,event_2,NetGMV,Unpaid_visits,Paid_visits
0,2022-06-06,Brand_A,5527.640078,0.000000,0,0,2647.596355,3260.875409,1007.683259
1,2022-06-06,Brand_B,8849.257500,8063.918386,0,0,682.406280,2085.216229,7504.890878
2,2022-06-13,Brand_A,6692.655692,0.000000,0,0,5020.823907,5043.292039,7341.265428
3,2022-06-13,Brand_B,9073.817994,9354.014585,0,0,3753.104897,8510.698145,2491.637379
4,2022-06-20,Brand_A,7124.016733,0.000000,0,0,6184.322132,5089.360559,69.521858
5,2022-06-20,Brand_B,7867.854558,5608.112521,0,0,3329.279953,4816.856815,1001.836097
6,2022-06-27,Brand_A,7725.169902,0.000000,0,0,5446.374631,8323.790786,8084.290324
7,2022-06-27,Brand_B,9712.332359,11760.981800,0,0,7544.192188,7051.240217,3931.021422
8,2022-07-04,Brand_A,8545.792935,0.000000,0,0,10058.970814,2281.596766,3152.531748
9,2022-07-04,Brand_B,6747.884370,6774.114961,0,0,2359.259385,6143.394347,5236.808509


# Export Enhanced Dataset

In [16]:
# Export the enhanced dataset
output_path = "data/test_geo_model_dataset.csv"
df_enhanced.to_csv(output_path, index=False)

print(f"Dataset successfully exported to: {output_path}")
print(f"Rows: {len(df_enhanced)}")
print(f"Columns: {list(df_enhanced.columns)}")

Dataset successfully exported to: data/test_geo_model_dataset.csv
Rows: 318
Columns: ['date', 'Groups', 'Spend_channel_1', 'Spend_channel_2', 'event_1', 'event_2', 'NetGMV', 'Unpaid_visits', 'Paid_visits']
